# 9장 어텐션 헤드 직접 계산하기

- GPU가 없어도 됩니다.
- 셀은 위에서부터 차례로 실행합니다(Shift + Enter).

데이터: [Tiny-Ko-Stories](https://huggingface.co/datasets/psymon/Tiny-Ko-Stories) (psymon, CC BY 4.0) · 모델: [JosephLee/modu-llm](https://huggingface.co/JosephLee/modu-llm) (CC BY 4.0)

## 실습 9-1 실습 준비

In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm-v2
import torch
from torch import nn
from torch.nn import functional as F
from modu_llm import Vocab, GPTConfig, Embeddings, show

torch.set_grad_enabled(False)      # 이 장에서는 학습하지 않으므로 기울기 계산을 끕니다

## 실습 9-2 레시피를 계산하는 부품 nn.Linear

**먼저 예상해 보세요.** 글자 B의 출력2는 얼마일까요?

In [ ]:
recipe = nn.Linear(3, 2, bias=False)       # 숫자 3개를 받아 숫자 2개를 만드는 부품
print("레시피 표 모양:", recipe.weight.shape)
recipe.weight[:] = torch.tensor([[0.5, 1.0, 0.0],      # 출력1의 레시피
                                 [-1.0, 0.0, 1.0]])    # 출력2의 레시피
a = torch.tensor([[1.0, 2.0, 0.0],         # 글자 A의 숫자 3개
                  [0.0, 1.0, 3.0]])        # 글자 B의 숫자 3개
show(recipe(a), rows=["글자 A", "글자 B"], cols=["출력1", "출력2"])

## 실습 9-3 예문의 숫자 카드

In [ ]:
vocab = Vocab.build()
sentence = "토끼가 숲에 가요"
idx = torch.tensor(vocab.encode(sentence))
labels = list(sentence.replace(" ", "␣"))   # 표의 줄·칸 이름

torch.manual_seed(42)
cfg = GPTConfig(n_embd=16)
emb = Embeddings(cfg)
x = emb(idx)
print("글자 번호:", idx.tolist())
print("x 모양:", x.shape)

## 실습 9-4 질문 쪽지·명찰·메모 만들기

In [ ]:
head_size = 8
query = nn.Linear(16, head_size, bias=False)   # 질문 쪽지를 만드는 부품
key = nn.Linear(16, head_size, bias=False)     # 명찰을 만드는 부품
value = nn.Linear(16, head_size, bias=False)   # 메모를 만드는 부품

q = query(x)
k = key(x)
v = value(x)
print("q:", q.shape)
print("k:", k.shape)
print("v:", v.shape)

## 실습 9-5 같은 자리끼리 곱해서 모두 더하기

**먼저 예상해 보세요.** '가'의 질문 쪽지 [2, 0, 0, 2]와 '토'의 명찰 [0, 0, 1, 2]를 같은 자리끼리 곱하면 어떤 숫자 네 개가 나올까요?

In [ ]:
q3 = torch.tensor([2.0, 0.0, 0.0, 2.0])    # '가'의 질문 쪽지
k1 = torch.tensor([0.0, 0.0, 1.0, 2.0])    # '토'의 명찰
k2 = torch.tensor([1.0, 0.0, 0.0, 0.0])    # '끼'의 명찰
k3 = torch.tensor([0.0, 1.0, 1.0, 0.0])    # '가'의 명찰

show([q3, k1, q3 * k1], rows=["가의 질문 쪽지", "토의 명찰", "같은 자리끼리 곱하기"])
print("모두 더하기:", (q3 * k1).sum())
print("@로 한 번에:", q3 @ k1, q3 @ k2, q3 @ k3)

## 실습 9-6 표 뒤집기: 전치

**먼저 예상해 보세요.** 뒤집은 표에서 '끼'의 명찰 [1, 0, 0, 0]은 어디에 놓일까요?

In [ ]:
names = ["토", "끼", "가"]
K = torch.stack([k1, k2, k3])    # 명찰 세 장을 쌓은 표
show(K, rows=names, mark_rows=0)
show(K.T, cols=names, mark_cols=0)
print(K.shape, "→", K.T.shape)

## 실습 9-7 행렬 곱셈으로 점수 표 만들기

**먼저 예상해 보세요.** 점수 표 S의 셋째 줄('가'의 질문 쪽지 줄)에는 어떤 숫자 세 개가 들어 있을까요?

In [ ]:
Q = torch.tensor([[0.0, 2.0, 0.0, 0.0],      # '토'의 질문 쪽지
                  [0.0, 0.0, -2.0, 2.0],     # '끼'의 질문 쪽지
                  [2.0, 0.0, 0.0, 2.0]])     # '가'의 질문 쪽지
S = Q @ K.T
print(S.shape)
show(S, rows=["토의 질문 쪽지", "끼의 질문 쪽지", "가의 질문 쪽지"],
     cols=["토의 명찰", "끼의 명찰", "가의 명찰"], mark_rows=2)

## 실습 9-8 nn.Linear도 행렬 곱셈입니다

In [ ]:
print("모양:", a.shape, "@", recipe.weight.T.shape, "→", (a @ recipe.weight.T).shape)
print("nn.Linear와 같은가요?", torch.allclose(recipe(a), a @ recipe.weight.T))

## 실습 9-9 예문 아홉 글자의 점수 표

**먼저 예상해 보세요.** q와 k의 모양은 (9, 8)입니다. q @ k.T의 모양은 어떻게 될까요?

In [ ]:
scores = q @ k.T                      # ① 점수: 질문 쪽지와 명찰의 내적
print("점수 표 모양:", scores.shape)
show(scores, rows=labels, cols=labels, mark_rows=2)

## 실습 9-10 ④ 비율: 점수를 비율로 바꾸는 함수, 소프트맥스

In [ ]:
s = S[2]                          # '가'의 점수 [4, 2, 0]
ex = torch.exp(s)                 # 점수만큼 e를 거듭제곱해 키우기
show([s, ex, ex / ex.sum(), F.softmax(s, dim=-1)],
     rows=["점수", "키운 값", "합으로 나누기", "softmax"], cols=names)

## 실습 9-11 ② 나누기: 비율이 한쪽으로 쏠리지 않게

**먼저 예상해 보세요.** 점수를 10배로 키우면 비율은 고르게 퍼질까요, 한쪽으로 몰릴까요?

In [ ]:
show([F.softmax(s * 10, dim=-1), F.softmax(s, dim=-1), F.softmax(s / 2, dim=-1)],
     rows=["점수 10배", "점수 그대로", "점수를 2로 나누기"], cols=names)

## 실습 9-12 ③ 가리기: 마스크

**먼저 예상해 보세요.** 뒤 글자를 가리면 '끼'는 어느 글자들에게 비율을 나눠 줄까요?

In [ ]:
mask = torch.tril(torch.ones(3, 3))
masked = (S / 2).masked_fill(mask == 0, float("-inf"))
W = F.softmax(masked, dim=-1)
show(mask, rows=names, cols=names)
show(masked, rows=names, cols=names)
show(W, rows=names, cols=names)
print("줄마다 합:", W.sum(dim=-1))

## 실습 9-13 예문 아홉 글자에 적용하기

In [ ]:
scores = scores / head_size ** 0.5    # ② 나누기
show(scores[2], rows=["가"], cols=labels)

## 실습 9-14 예문 아홉 글자에 적용하기

**먼저 예상해 보세요.** 비율 표의 첫째 줄, 곧 첫 글자 '토'의 줄은 어떤 모양일까요?

In [ ]:
T = len(idx)
mask = torch.tril(torch.ones(T, T))
scores = scores.masked_fill(mask == 0, float("-inf"))   # ③ 가리기
weights = F.softmax(scores, dim=-1)                       # ④ 비율
print("줄마다 합:", weights.sum(dim=-1).round(decimals=2))
show(weights, rows=labels, cols=labels, mark_rows=2)

## 실습 9-15 ⑤ 모으기

**먼저 예상해 보세요.** 첫 글자 '토'가 모아 온 결과는 무엇과 같을까요?

In [ ]:
out = weights @ v                      # ⑤ 모으기
print("out 모양:", out.shape)
print("'토'의 결과와 '토'의 메모가 같은가요?", torch.allclose(out[0], v[0]))

## 실습 9-16 패키지 부품과 결과 비교하기

In [ ]:
from modu_llm import Head

head = Head(cfg, head_size)
head.query = query
head.key = key
head.value = value
print("결과가 같은가요?", torch.allclose(head(x), out))
print("비율 표가 같은가요?", torch.allclose(head.weights, weights))